# 03 - Preguntas de negocio

## (a) ¿Cuántos buques distintos transmitieron cada día?

Antes de esta pregunta se eliminan duplicados exactos y se exige un MMSI numérico de nueve dígitos, como indica D-07. No se usa `cache()` ni `persist()`.

In [0]:
# Carga unificada de los siete archivos AIS con esquema explícito.
# Se agrega la columna ingestion_date a partir de la ruta del archivo para conservar la fecha de origen de cada posición y habilitar análisis diarios sobre el período 01-07 de junio de 2023.
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

ROOT = "/Volumes/oceanwatch_g06/landing/raw_ais"
DATES = ["2023-06-01","2023-06-02","2023-06-03","2023-06-04","2023-06-05","2023-06-06","2023-06-07"]
SCHEMA = StructType([StructField("MMSI",StringType(),True),StructField("BaseDateTime",StringType(),True),StructField("LAT",DoubleType(),True),StructField("LON",DoubleType(),True),StructField("SOG",DoubleType(),True),StructField("COG",DoubleType(),True),StructField("Heading",DoubleType(),True),StructField("VesselName",StringType(),True),StructField("IMO",StringType(),True),StructField("CallSign",StringType(),True),StructField("VesselType",IntegerType(),True),StructField("Status",IntegerType(),True),StructField("Length",DoubleType(),True),StructField("Width",DoubleType(),True),StructField("Draft",DoubleType(),True),StructField("Cargo",IntegerType(),True),StructField("TransceiverClass",StringType(),True)])
paths = [f"{ROOT}/extracted/ingestion_date={d}/AIS_{d.replace('-', '_')}.csv" for d in DATES]
raw_ais=(spark.read.option("header","true").option("enforceSchema","false").option("mode","FAILFAST").schema(SCHEMA).csv(paths).withColumn("ingestion_date",F.regexp_extract(F.col("_metadata.file_path"),r"ingestion_date=(\d{4}-\d{2}-\d{2})",1)))

In [0]:
# Conjunto canónico virtual D-07: no persiste ni escribe resultados intermedios.
# Construcción del conjunto canónico de análisis. Se eliminan duplicados exactos y se comparan los conteos de embarcaciones distintas por día antes y después de excluir MMSI que no cumplen el formato AIS estándar de 9 dígitos.
canonical_ais=raw_ais.dropDuplicates(SCHEMA.fieldNames())
valid_mmsi_ais=canonical_ais.filter(F.col("MMSI").rlike(r"^[0-9]{9}$"))
with_invalid=canonical_ais.groupBy("ingestion_date").agg(F.countDistinct("MMSI").alias("mmsi_distintos_sin_excluir"))
valid_only=valid_mmsi_ais.groupBy("ingestion_date").agg(F.countDistinct("MMSI").alias("mmsi_distintos_validos"))
display(with_invalid.join(valid_only,"ingestion_date").withColumn("efecto_exclusion",F.col("mmsi_distintos_sin_excluir")-F.col("mmsi_distintos_validos")).orderBy("ingestion_date"))

ingestion_date,mmsi_distintos_sin_excluir,mmsi_distintos_validos,efecto_exclusion
2023-06-01,20448,20422,26
2023-06-02,21153,21086,67
2023-06-03,20505,20456,49
2023-06-04,19720,19698,22
2023-06-05,19615,19597,18
2023-06-06,19717,19691,26
2023-06-07,20086,20056,30


In [0]:
# Comparación entre conteos exactos y aproximados de embarcaciones distintas.
# Se evalúa el error de approx_count_distinct frente al valor exacto para determinar si una aproximación sería suficiente en escenarios de mayor escala.
exact_by_day=valid_mmsi_ais.groupBy("ingestion_date").agg(F.countDistinct("MMSI").alias("count_distinct_exacto"))
approx_by_day=valid_mmsi_ais.groupBy("ingestion_date").agg(F.approx_count_distinct("MMSI").alias("approx_count_distinct"))
comparison=(exact_by_day.join(approx_by_day,"ingestion_date").withColumn("diferencia_absoluta",F.abs(F.col("approx_count_distinct")-F.col("count_distinct_exacto"))).withColumn("diferencia_porcentaje",F.round(100*F.col("diferencia_absoluta")/F.col("count_distinct_exacto"),4)).orderBy("ingestion_date"))
display(comparison)

ingestion_date,count_distinct_exacto,approx_count_distinct,diferencia_absoluta,diferencia_porcentaje
2023-06-01,20422,21782,1360,6.6595
2023-06-02,21086,23028,1942,9.2099
2023-06-03,20456,20680,224,1.095
2023-06-04,19698,20275,577,2.9292
2023-06-05,19597,20358,761,3.8832
2023-06-06,19691,20288,597,3.0318
2023-06-07,20056,21184,1128,5.6243


In [0]:
# Comparación de planes físicos de ejecución.
# Se documentan los operadores utilizados por Spark para los conteos exactos y aproximados, con el fin de justificar la selección de la estrategia usada en el análisis de embarcaciones distintas por día.
print("=== PLAN countDistinct exacto ===")
exact_by_day.explain("formatted")
print("=== PLAN approx_count_distinct ===")
approx_by_day.explain("formatted")

=== PLAN countDistinct exacto ===
== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   PhotonResultStage (24)
   +- PhotonColumnarToRow (23)
      +- PhotonGroupingAgg (22)
         +- PhotonShuffleExchangeSource (21)
            +- PhotonShuffleMapStage (20)
               +- PhotonShuffleExchangeSink (19)
                  +- PhotonGroupingAgg (18)
                     +- PhotonGroupingAgg (17)
                        +- PhotonShuffleExchangeSource (16)
                           +- PhotonShuffleMapStage (15)
                              +- PhotonShuffleExchangeSink (14)
                                 +- PhotonGroupingAgg (13)
                                    +- PhotonGroupingAgg (12)
                                       +- PhotonShuffleExchangeSource (11)
                                          +- PhotonShuffleMapStage (10)
                                             +- PhotonShuffleExchangeSink (9)
                                                +- PhotonG

## Decisión para producción

Se usa `countDistinct`: la cardinalidad diaria ronda 20 mil MMSI y el resultado exacto es más defendible. Ambos planes leen el mismo volumen base; el menor estado de HyperLogLog no compensa introducir error en un cálculo diario de cardinalidad baja. `approx_count_distinct` queda como alternativa para una escala mayor o un SLA medido.

## (b) ¿Qué tipos de buque generan más tráfico?

El catálogo proviene de NOAA Marine Cadastre, `VesselTypeCodes2018.pdf`, y se contrasta con la guía AIS USCG basada en ITU-R M.1371. D-07 no filtra MMSI aquí porque la métrica agrega posiciones por tipo, sin identificar ni agrupar buques individuales. La deduplicación exacta se mantiene como regla transversal.

In [0]:
# Catálogo de tipos de embarcación basado en las clasificaciones AIS utilizadas por NOAA y USCG. Se utiliza posteriormente para traducir los códigos VesselType a descripciones comprensibles en los análisis.
VESSEL_TYPE_CATALOG = [
    (30, "Fishing"), (31, "Towing"), (32, "Towing: tow >200m or breadth >25m"),
    (33, "Dredging or underwater operations"), (34, "Diving operations"),
    (35, "Military operations"), (36, "Sailing"), (37, "Pleasure craft"),
    (40, "High speed craft"), (50, "Pilot vessel"), (51, "Search and rescue"),
    (52, "Tug"), (53, "Port tender"), (54, "Anti-pollution equipment"),
    (55, "Law enforcement"), (56, "Local vessel assignment (spare)"),
    (57, "Local vessel assignment (spare)"), (58, "Medical transport"),
    (59, "Noncombatant ship"), (60, "Passenger"), (70, "Cargo"),
    (71, "Cargo: hazard A"), (72, "Cargo: hazard B"), (73, "Cargo: hazard C"),
    (74, "Cargo: hazard D"), (79, "Cargo: no additional information"),
    (80, "Tanker"), (81, "Tanker: hazard A"), (82, "Tanker: hazard B"),
    (83, "Tanker: hazard C"), (84, "Tanker: hazard D"), (89, "Tanker: no additional information"),
    (90, "Other type"), (99, "Other type: no additional information"),
]
vessel_type_catalog = spark.createDataFrame(VESSEL_TYPE_CATALOG, "VesselType int, vessel_type_description string")

In [0]:
# SOG=102.3 es “no disponible”; se muestra con y sin excluirlo. Null SOG se ignora por avg.
traffic_by_type = (canonical_ais.groupBy("VesselType").agg(
    F.count("*").alias("posiciones"),
    F.avg("SOG").alias("sog_media_con_sentinela"),
    F.avg(F.when(F.col("SOG") != 102.3, F.col("SOG"))).alias("sog_media_sin_sentinela"),
    F.sum((F.col("SOG") == 102.3).cast("long")).alias("registros_sog_102_3"),
).join(vessel_type_catalog, "VesselType", "left")
 .withColumn("vessel_type_description", F.coalesce("vessel_type_description", F.lit("No disponible/reservado/no catalogado")))
 .withColumn("efecto_sentinela_nudos", F.round(F.col("sog_media_con_sentinela") - F.col("sog_media_sin_sentinela"), 4))
 .orderBy(F.desc("posiciones")))
display(traffic_by_type.limit(10))

VesselType,posiciones,sog_media_con_sentinela,sog_media_sin_sentinela,registros_sog_102_3,vessel_type_description,efecto_sentinela_nudos
31,16557760,1.8987562508447051,1.6769118810882848,36505,Towing,0.2218
37,14476457,1.6422205999709396,1.3477206606019478,42231,Pleasure craft,0.2945
60,4795082,4.43393086916938,4.108974301835665,15869,Passenger,0.325
30,4010116,2.4661458671022696,2.2447832224179893,8872,Fishing,0.2214
36,3876193,1.9792172887162593,1.6257060432634487,13611,Sailing,0.3535
90,3848064,2.0281866933608517,1.7652278439886602,10065,Other type,0.263
70,3837757,6.369869405488288,6.330083630374322,1591,Cargo,0.0398
52,1921509,2.093071799299407,1.7235867220349639,7059,Tug,0.3695
80,1789094,5.874558910823276,5.861945540735694,234,Tanker,0.0126
57,1323745,1.9109511272941027,1.9042770143624308,88,Local vessel assignment (spare),0.0067


In [0]:
print("=== PLAN 03b: tráfico y velocidad por VesselType ===")
traffic_by_type.explain("formatted")

=== PLAN 03b: tráfico y velocidad por VesselType ===
== Physical Plan ==
AdaptiveSparkPlan (28)
+- == Initial Plan ==
   PhotonResultStage (27)
   +- PhotonColumnarToRow (26)
      +- PhotonSort (25)
         +- PhotonShuffleExchangeSource (24)
            +- PhotonShuffleMapStage (23)
               +- PhotonShuffleExchangeSink (22)
                  +- PhotonProject (21)
                     +- PhotonBroadcastHashJoin LeftOuter (20)
                        :- PhotonGroupingAgg (14)
                        :  +- PhotonShuffleExchangeSource (13)
                        :     +- PhotonShuffleMapStage (12)
                        :        +- PhotonShuffleExchangeSink (11)
                        :           +- PhotonGroupingAgg (10)
                        :              +- PhotonGroupingAgg (9)
                        :                 +- PhotonShuffleExchangeSource (8)
                        :                    +- PhotonShuffleMapStage (7)
                        :                   

**Costo.** Como en 03a, el plan escanea los siete CSV y hace la deduplicación global D-07. Luego agrega por `VesselType`, de cardinalidad baja, sin requerir el shuffle adicional de pares `(día, MMSI)` de `countDistinct` exacto. `SOG=102.3` se excluye del promedio, no del conteo de posiciones.

## (c) Diez buques con mayor distancia semanal

Esta sección reutiliza D-04: MMSI válido, timestamp y coordenadas calculables, `0 < gap <= 2 h` y velocidad implícita `<= 60 kn`. D-07 aplica completo porque la métrica agrupa por MMSI.

In [0]:
from pyspark.sql.window import Window

# D-07 ya eliminó copias exactas y MMSI no conformes antes de crear trayectorias.

# Cálculo de distancia recorrida por embarcación.
# Se construyen trayectorias a partir de posiciones AIS consecutivas, se estima la distancia geodésica mediante la fórmula de Haversine y se excluyen pares de observaciones con intervalos temporales o velocidades incompatibles con una trayectoria físicamente plausible.

events = valid_mmsi_ais.select("MMSI", "VesselName", "VesselType", "LAT", "LON", F.to_timestamp("BaseDateTime", "yyyy-MM-dd'T'HH:mm:ss").alias("event_ts"))
window_mmsi = Window.partitionBy("MMSI").orderBy("event_ts")
pairs = (events.withColumn("prev_ts", F.lag("event_ts").over(window_mmsi))
    .withColumn("prev_lat", F.lag("LAT").over(window_mmsi))
    .withColumn("prev_lon", F.lag("LON").over(window_mmsi))
    .filter(F.col("prev_ts").isNotNull())
    .withColumn("gap_hours", (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / F.lit(3600.0))
    .withColumn("a", F.pow(F.sin((F.radians("LAT") - F.radians("prev_lat")) / 2), 2) + F.cos(F.radians("prev_lat")) * F.cos(F.radians("LAT")) * F.pow(F.sin((F.radians("LON") - F.radians("prev_lon")) / 2), 2))
    .withColumn("distance_km", F.lit(6371.0088) * 2 * F.asin(F.sqrt("a")))
    .withColumn("implied_knots", F.col("distance_km") / F.col("gap_hours") / F.lit(1.852))
    .withColumn("d04_eligible", (F.col("gap_hours") > 0) & (F.col("gap_hours") <= 2) & (F.col("implied_knots") <= 60)))

# Los pares no elegibles se retienen como auditoría, pero no suman distancia.
distance_by_mmsi = pairs.groupBy("MMSI").agg(
    F.sum(F.when(F.col("d04_eligible"), F.col("distance_km")).otherwise(F.lit(0.0))).alias("distance_km"),
    F.sum(F.col("d04_eligible").cast("long")).alias("pares_elegibles"),
    F.sum((~F.col("d04_eligible")).cast("long")).alias("pares_excluidos"),
    F.sum(F.when(F.col("d04_eligible"), F.col("gap_hours")).otherwise(F.lit(0.0))).alias("horas_elegibles"),
)
vessel_info = valid_mmsi_ais.groupBy("MMSI").agg(
    F.first("VesselName", ignorenulls=True).alias("VesselName"),
    F.first("VesselType", ignorenulls=True).alias("VesselType"),
    F.countDistinct("VesselName").alias("nombres_distintos"),
    F.countDistinct("VesselType").alias("tipos_distintos"),
)
top10_distance = (distance_by_mmsi.join(vessel_info, "MMSI", "left")
    .withColumn(
        "velocidad_media_implicita_kn",
        F.when(
        F.col("horas_elegibles") > 0,
        F.col("distance_km") / F.col("horas_elegibles") / F.lit(1.852)
        )
    )
    .withColumn("velocidad_semana_calendario_kn", F.col("distance_km") / F.lit(168 * 1.852))
    .orderBy(F.desc("distance_km")).limit(10))
display(top10_distance.select("MMSI", "VesselName", "VesselType", F.round("distance_km", 2).alias("distance_km"), F.round("velocidad_media_implicita_kn", 2).alias("velocidad_media_implicita_kn"), F.round("velocidad_semana_calendario_kn", 2).alias("velocidad_semana_calendario_kn"), "pares_elegibles", "pares_excluidos", "nombres_distintos", "tipos_distintos"))

MMSI,VesselName,VesselType,distance_km,velocidad_media_implicita_kn,velocidad_semana_calendario_kn,pares_elegibles,pares_excluidos,nombres_distintos,tipos_distintos
367638030,JUSTIN PAUL ECKSTEIN,31,5770.92,18.61,18.55,2865,8,1,1
367438630,A STEVE CROWLEY,31,3749.18,12.12,12.05,8324,10,1,1
366991412,USNS APALACHICOLA,35,3638.87,11.7,11.7,5301,0,1,1
368097380,KOALAFIED CRUISER,60,3632.87,16.46,11.68,6794,7,1,1
366941210,LAKE EXPRESS,60,3614.35,11.76,11.62,6293,224,1,1
316011408,COASTAL INSPIRATION,60,3499.1,11.25,11.25,7208,0,1,1
316001245,QUEEN OF ALBERNI,60,3428.77,11.03,11.02,7197,4,1,1
316001251,QUEEN OF COWICHAN,60,3222.26,10.36,10.36,6657,2,1,1
368067110,LADY SWIFT,40,3139.82,10.1,10.09,6260,1,1,1
316001257,QUEEN OF OAK BAY,60,3099.4,10.11,9.96,5998,3,1,1


In [0]:
display(pairs.agg(F.count("*").alias("pares_totales_D07"), F.sum(F.col("d04_eligible").cast("long")).alias("pares_elegibles_D04"), F.sum((~F.col("d04_eligible")).cast("long")).alias("pares_excluidos_D04")))
display(events.groupBy("MMSI").count().agg(F.count("*").alias("mmsi"), F.expr("percentile_approx(count, array(0.5, 0.95, 0.99))").alias("p50_p95_p99_posiciones"), F.max("count").alias("max_posiciones")))
print("=== PLAN 03c: ventana por MMSI ===")
top10_distance.explain("formatted")

pares_totales_D07,pares_elegibles_D04,pares_excluidos_D04
60450496,60369977,80519


mmsi,p50_p95_p99_posiciones,max_posiciones
31780,"List(957, 8197, 8607)",9581


=== PLAN 03c: ventana por MMSI ===
== Physical Plan ==
AdaptiveSparkPlan (45)
+- == Initial Plan ==
   PhotonResultStage (44)
   +- PhotonColumnarToRow (43)
      +- PhotonTopK (42)
         +- PhotonShuffleExchangeSource (41)
            +- PhotonShuffleMapStage (40)
               +- PhotonShuffleExchangeSink (39)
                  +- PhotonTopK (38)
                     +- PhotonProject (37)
                        +- PhotonShuffledHashJoin LeftOuter (36)
                           :- PhotonGroupingAgg (19)
                           :  +- PhotonProject (18)
                           :     +- PhotonProject (17)
                           :        +- PhotonFilter (16)
                           :           +- PhotonWindow (15)
                           :              +- PhotonSort (14)
                           :                 +- PhotonShuffleExchangeSource (13)
                           :                    +- PhotonShuffleMapStage (12)
                           :            

## (d) ¿Dónde se concentra el tráfico?

D-07 elimina duplicados exactos. Los MMSI no conformes no se filtran porque se cuentan posiciones por celda, no buques individuales. H3 se calcula con funciones nativas de Databricks y conserva el plan Photon.

In [0]:
import hashlib
import os
import requests

# Carga del World Port Index (WPI) como conjunto de referencia geográfica.
# Este catálogo se utilizará posteriormente para contextualizar las celdas H3 más transitadas y asociarlas con puertos cercanos cuando sea posible.

WPI_URL = "https://msi.nga.mil/api/publications/download?type=view&key=16920959/SFH00000/UpdatedPub150.csv"
WPI_VOLUME = "/Volumes/oceanwatch_g06/reference/world_port_index"
WPI_PATH = f"{WPI_VOLUME}/UpdatedPub150.csv"
spark.sql("CREATE VOLUME IF NOT EXISTS oceanwatch_g06.reference.world_port_index COMMENT 'World Port Index (NGA Pub. 150); referencia para asociación de celdas H3 a puertos.'")
os.makedirs(WPI_VOLUME, exist_ok=True)
if not os.path.exists(WPI_PATH):
    response = requests.get(WPI_URL, timeout=120)
    response.raise_for_status()
    with open(WPI_PATH, "wb") as output:
        output.write(response.content)
with open(WPI_PATH, "rb") as source:
    print(f"WPI bytes={os.path.getsize(WPI_PATH)}; sha256={hashlib.sha256(source.read()).hexdigest()}")

ports = (spark.read.option("header", "true").option("multiLine", "true").option("escape", '\"').csv(WPI_PATH)
    .select(F.col("World Port Index Number").cast("string").alias("wpi_id"), F.col("Main Port Name").alias("port_name"), F.col("Country Code").alias("country"), F.col("Latitude").cast("double").alias("port_lat"), F.col("Longitude").cast("double").alias("port_lon"))
    .filter(F.col("port_lat").isNotNull() & F.col("port_lon").isNotNull()))
print(f"Puertos WPI con coordenadas: {ports.count()}")

WPI bytes=3508891; sha256=315644f1e77966291633145fd351c2762b37702d115926b9ed074b39e8e21667
Puertos WPI con coordenadas: 3807


In [0]:
spark.read.option("header","true").csv(WPI_PATH) \
.select("Latitude","Longitude") \
.filter(
~F.col("Latitude").rlike(r"^-?\d+(\.\d+)?$")
|
~F.col("Longitude").rlike(r"^-?\d+(\.\d+)?$")
) \
.count()

0

In [0]:
# Identificación de las celdas H3 más transitadas.
# Cada posición AIS se asigna a una celda hexagonal H3 de resolución 8, se contabiliza el tráfico por celda y posteriormente se relacionan los centroides de las celdas más frecuentes con puertos del World Port Index.

h3_positions = canonical_ais.select(F.expr("h3_longlatash3(LON, LAT, 8)").alias("h3_r8"))
top_h3 = h3_positions.groupBy("h3_r8").count().withColumnRenamed("count", "posiciones").orderBy(F.desc("posiciones")).limit(10)
top_cells = (top_h3.withColumn("h3_hex", F.expr("h3_h3tostring(h3_r8)"))
    .withColumn("center_wkt", F.expr("h3_centeraswkt(h3_r8)"))
   # .withColumn("center_lon", F.regexp_extract("center_wkt", r"POINT\\s*\\(([-0-9.]+)\\s+[-0-9.]+\\)", 1).cast("double"))
   # .withColumn("center_lat", F.regexp_extract("center_wkt", r"POINT\\s*\\([-0-9.]+\\s+([-0-9.]+)\\)", 1).cast("double"))
   .withColumn("center_lon",F.split(F.regexp_replace(F.regexp_replace("center_wkt", "POINT\\(", ""),"\\)", "")," ").getItem(0).cast("double"))
   .withColumn("center_lat",F.split(F.regexp_replace(F.regexp_replace("center_wkt", "POINT\\(", ""),"\\)", "")," ").getItem(1).cast("double"))
)

# Solo se cruzan 10 centroides con la referencia pequeña WPI; el broadcast se verifica en el plan.
port_candidates = (top_cells.crossJoin(F.broadcast(ports))
    .withColumn("a", F.pow(F.sin((F.radians("port_lat") - F.radians("center_lat")) / 2), 2) + F.cos(F.radians("center_lat")) * F.cos(F.radians("port_lat")) * F.pow(F.sin((F.radians("port_lon") - F.radians("center_lon")) / 2), 2))
    .withColumn("distancia_puerto_km", F.lit(12742.0176) * F.asin(F.sqrt("a")))
    .withColumn("puerto_etiqueta", F.concat_ws(" | ", "port_name", "country", "wpi_id")))

top_cells_with_ports = (port_candidates.groupBy("h3_r8", "h3_hex", "posiciones", "center_lat", "center_lon").agg(
    F.round(F.min("distancia_puerto_km"), 3).alias("puerto_mas_cercano_km"),
    F.sort_array(F.collect_list(F.when(F.col("distancia_puerto_km") <= 1, F.col("puerto_etiqueta")))).alias("puertos_1km"),
    F.sort_array(F.collect_list(F.when(F.col("distancia_puerto_km") <= 2, F.col("puerto_etiqueta")))).alias("puertos_2km"),
    F.sort_array(F.collect_list(F.when(F.col("distancia_puerto_km") <= 5, F.col("puerto_etiqueta")))).alias("puertos_5km"),
).withColumn("asociada_1km", F.size("puertos_1km") > 0)
 .withColumn("asociada_2km", F.size("puertos_2km") > 0)
 .withColumn("asociada_5km", F.size("puertos_5km") > 0)
 .orderBy(F.desc("posiciones")))
display(top_cells_with_ports)

h3_r8,h3_hex,posiciones,center_lat,center_lon,puerto_mas_cercano_km,puertos_1km,puertos_2km,puertos_5km,asociada_1km,asociada_2km,asociada_5km
613207896999591935,8828d555a1fffff,235260,47.629802764,-122.387803097,5.259,List(),List(),List(),false,false,false
613207893094694911,8828d54715fffff,184408,47.679754638,-122.407095935,9.882,List(),List(),List(),false,false,false
613222103734288383,8829a411d7fffff,180097,32.717595785,-117.231275797,4.486,List(),List(),List(San Diego | United States | 16010.0),false,false,true
613207632246734847,8828d17b59fffff,123782,48.758286123,-122.503656139,0.96,List(Bellingham | United States | 18050.0),List(Bellingham | United States | 18050.0),List(Bellingham | United States | 18050.0),true,true,true
613221934181646335,8829a19a35fffff,115457,33.980380856,-118.450953433,7.269,List(),List(),List(),false,false,false
613693378015526911,88446e0359fffff,114643,29.966252957,-93.861539332,9.31,List(),List(),List(),false,false,false
613212098154987519,8829127827fffff,109522,34.244790395,-119.263224298,4.667,List(),List(),List(Ventura | United States | 16140.0),false,false,true
613696897732837375,8844a13b51fffff,103931,26.099539866,-80.16161502,4.489,List(),List(),List(Port Everglades | United States | 8630.0),false,false,true
613207893184872447,8828d5476bfffff,101036,47.659698393,-122.374342684,7.315,List(),List(),List(),false,false,false
613222103686053887,8829a411a9fffff,100848,32.725239602,-117.19004354,1.141,List(),List(San Diego | United States | 16010.0),List(San Diego | United States | 16010.0),false,true,true


In [0]:
# Análisis de sensibilidad de la asociación puerto-celda.
# Se compara cuántas de las celdas H3 más transitadas pueden asociarse a un puerto del World Port Index utilizando radios de 1 km, 2 km y 5 km, con el fin de evaluar qué tan dependientes son las conclusiones del umbral espacial elegido.

sensitivity_d03 = top_cells_with_ports.agg(
    F.sum(F.col("asociada_1km").cast("int")).alias("celdas_asociadas_1km"),
    F.sum(F.col("asociada_2km").cast("int")).alias("celdas_asociadas_2km"),
    F.sum(F.col("asociada_5km").cast("int")).alias("celdas_asociadas_5km"),
    F.sum(F.when(F.col("asociada_1km"), F.col("posiciones")).otherwise(0)).alias("posiciones_1km"),
    F.sum(F.when(F.col("asociada_2km"), F.col("posiciones")).otherwise(0)).alias("posiciones_2km"),
    F.sum(F.when(F.col("asociada_5km"), F.col("posiciones")).otherwise(0)).alias("posiciones_5km"),
    F.sum((F.col("asociada_1km") != F.col("asociada_5km")).cast("int")).alias("celdas_que_cambian_1_a_5km"),
    F.sum((F.col("asociada_2km") != F.col("asociada_5km")).cast("int")).alias("celdas_que_cambian_2_a_5km"),
)
display(sensitivity_d03)
print("=== PLAN 03d: H3 nativo y WPI broadcast ===")
sensitivity_d03.explain("formatted")

celdas_asociadas_1km,celdas_asociadas_2km,celdas_asociadas_5km,posiciones_1km,posiciones_2km,posiciones_5km,celdas_que_cambian_1_a_5km,celdas_que_cambian_2_a_5km
1,2,5,123782,224630,618180,4,3


=== PLAN 03d: H3 nativo y WPI broadcast ===
== Physical Plan ==
AdaptiveSparkPlan (41)
+- == Initial Plan ==
   PhotonResultStage (40)
   +- PhotonColumnarToRow (39)
      +- PhotonAgg (38)
         +- PhotonGroupingAgg (37)
            +- PhotonProject (36)
               +- PhotonProject (35)
                  +- PhotonBroadcastNestedLoopJoin Cross BuildRight (34)
                     :- PhotonGlobalLimit (26)
                     :  +- PhotonSort (25)
                     :     +- PhotonShuffleExchangeSource (24)
                     :        +- PhotonShuffleMapStage (23)
                     :           +- PhotonShuffleExchangeSink (22)
                     :              +- PhotonLocalLimit (21)
                     :                 +- PhotonProject (20)
                     :                    +- PhotonProject (19)
                     :                       +- PhotonSort (18)
                     :                          +- PhotonShuffleExchangeSource (17)
                 

## (e) ¿Qué proporción transmitió los siete días y dónde están los visitantes?

D-07 aplica completo porque la pregunta agrupa por MMSI. No se usa `cache()` ni `persist()`.

In [0]:
# Se usa una fila por MMSI válido. first(VesselType) permite caracterizar visitantes y tipos_distintos deja trazabilidad de mensajes inconsistentes.
mmsi_day_profile = valid_mmsi_ais.groupBy("MMSI").agg(
    F.countDistinct("ingestion_date").alias("dias_transmitidos"),
    F.min("ingestion_date").alias("dia_unico"),
    F.first("VesselType", ignorenulls=True).alias("VesselType"),
    F.countDistinct("VesselType").alias("tipos_distintos"),
)
distribution_days = mmsi_day_profile.groupBy("dias_transmitidos").count().withColumnRenamed("count", "buques").orderBy("dias_transmitidos")
total_valid_week = mmsi_day_profile.count()
seven_day_vessels = mmsi_day_profile.filter(F.col("dias_transmitidos") == 7).count()
display(distribution_days.withColumn("porcentaje", F.round(100 * F.col("buques") / F.lit(total_valid_week), 4)))
print(f"MMSI válidos={total_valid_week}; 7 días={seven_day_vessels}; proporción={100 * seven_day_vessels / total_valid_week:.4f}%")

dias_transmitidos,buques,porcentaje
1,5964,18.7665
2,4361,13.7225
3,2888,9.0875
4,2200,6.9226
5,2002,6.2996
6,1709,5.3776
7,12656,39.8238


MMSI válidos=31780; 7 días=12656; proporción=39.8238%


In [0]:
one_day_visitors = mmsi_day_profile.filter(F.col("dias_transmitidos") == 1)
total_one_day = one_day_visitors.count()
visitor_types = (one_day_visitors.groupBy("VesselType").count().withColumnRenamed("count", "buques")
    .withColumn("porcentaje_visitantes", F.round(100 * F.col("buques") / F.lit(total_one_day), 4))
    .orderBy(F.desc("buques")))
visitor_days = (one_day_visitors.groupBy("dia_unico").count().withColumnRenamed("count", "buques")
    .withColumn("porcentaje_visitantes", F.round(100 * F.col("buques") / F.lit(total_one_day), 4))
    .orderBy(F.desc("buques")))
display(visitor_types)
display(visitor_days)

VesselType,buques,porcentaje_visitantes
37,2795,46.8645
36,1113,18.662
null,504,8.4507
30,365,6.1201
31,246,4.1247
70,178,2.9846
90,162,2.7163
60,126,2.1127
80,108,1.8109
0,73,1.224


dia_unico,buques,porcentaje_visitantes
2023-06-01,1326,22.2334
2023-06-07,1244,20.8585
2023-06-02,831,13.9336
2023-06-03,759,12.7264
2023-06-04,723,12.1227
2023-06-06,580,9.725
2023-06-05,501,8.4004


In [0]:
# Control explícito: cada MMSI aparece exactamente en una categoría de días transmitidos.
sanity_03e = distribution_days.agg(F.sum("buques").alias("suma_distribucion")).withColumn("mmsi_validos_semana", F.lit(total_valid_week))
display(sanity_03e.withColumn("control_ok", F.col("suma_distribucion") == F.col("mmsi_validos_semana")))
print("=== PLAN 03e: agregación inversa a 03a (por MMSI, luego por días) ===")
distribution_days.explain("formatted")

suma_distribucion,mmsi_validos_semana,control_ok
31780,31780,true


=== PLAN 03e: agregación inversa a 03a (por MMSI, luego por días) ===
== Physical Plan ==
AdaptiveSparkPlan (34)
+- == Initial Plan ==
   PhotonResultStage (33)
   +- PhotonColumnarToRow (32)
      +- PhotonSort (31)
         +- PhotonShuffleExchangeSource (30)
            +- PhotonShuffleMapStage (29)
               +- PhotonShuffleExchangeSink (28)
                  +- PhotonGroupingAgg (27)
                     +- PhotonShuffleExchangeSource (26)
                        +- PhotonShuffleMapStage (25)
                           +- PhotonShuffleExchangeSink (24)
                              +- PhotonGroupingAgg (23)
                                 +- PhotonGroupingAgg (22)
                                    +- PhotonShuffleExchangeSource (21)
                                       +- PhotonShuffleMapStage (20)
                                          +- PhotonShuffleExchangeSink (19)
                                             +- PhotonGroupingAgg (18)
                            